# pre

In [ ]:
import pickle

# Load list and dictionary from file
with open('result.pkl', 'rb') as f:
    names_baseline = pickle.load(f)
    status_baseline = pickle.load(f)

# library

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

from mistral_inference.transformer import Transformer
from mistral_inference.generate import generate
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from mistral_common.protocol.instruct.messages import UserMessage
from mistral_common.protocol.instruct.request import ChatCompletionRequest

import gradio as gr
import re

# llm

In [2]:
# 4s
mistral_models_path = ('../rag/mistral')
tokenizer = MistralTokenizer.from_file(f"{mistral_models_path}/tokenizer.model.v3")
model = Transformer.from_folder(mistral_models_path)

def llm(input):
    completion_request = ChatCompletionRequest(messages=[UserMessage(content=input)])
    tokens = tokenizer.encode_chat_completion(completion_request).tokens
    out_tokens, _ = generate([tokens], model, max_tokens=10000, temperature=0.0, eos_id=tokenizer.instruct_tokenizer.tokenizer.eos_id)
    result = tokenizer.instruct_tokenizer.tokenizer.decode(out_tokens[0])
    return result

# pdf read-into database

In [ ]:
# 20s
file_path = ('data.pdf')
loader = PyPDFLoader(file_path)
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=4000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)
pages = loader.load_and_split(text_splitter=text_splitter)

In [61]:
# table
pattern = r'Table \d+\.\d+\.'
table_list = []
other_list = []
for i in range(0, len(pages)):
    if re.search(pattern, pages[i].page_content):
        pages[i].metadata['label'] = 'Table'
        table_list.append(pages[i])
    else:
        pages[i].metadata['label'] = 'Text'
        other_list.append(pages[i])

In [ ]:
## species name
species_name_pattern = re.compile(r'species name', re.IGNORECASE)
species_list = []
#other_list = []
for i in range(0, len(pages)):
    if re.search(species_name_pattern, pages[i].page_content):
        pages[i].metadata['species'] = 'Yes'
        species_list.append(pages[i])
    else:
        pages[i].metadata['species'] = 'No'
        #other_list.append(pages[i])

In [63]:
## Conservation Status
status_pattern = pattern = r'Conservation Status'
status_list = []
#other_list = []
for i in range(0, len(pages)):
    if re.search(status_pattern, pages[i].page_content):
        #pages[i].metadata['status'] = 'Yes'
        status_list.append(pages[i])
    #else:
        #pages[i].metadata['status'] = 'No'
        #other_list.append(pages[i])

In [64]:
status_list

[Document(metadata={'source': 'data.pdf', 'page': 13, 'label': 'Text'}, page_content='21009_Reconstruction of CCKWW_Draft EIA Report 14 \nGLOSSARY \n \nAbundance: The number of a single species recorded at any given time period or location. \n \nBiodiversity: The variety of plant and animal life in the world, habitat or location, a high level \nof which is usually considered to be important and desirable. Biodiversity can be assessed at \nmore focused taxonomic groups such as “bird biodiversity”, in which case it is interchangeably \nwith “diversity”. \n \nConservation Status: A status given to a species that is threatened with becoming extinct \neither locally or globally. These species may be restricted to only a small area, show noticeable \ndecline in abundance over time, or have a historically low global population size. Assessments \ncan be made either at global level under the IUCN’s Red List of Threatened Species or at \nnational level (e.g., Singapore’s Red Data Book of Threat

In [65]:
## Conservation Status
status_pattern = pattern = r'Conservation Status'
status_list = []
#other_list = []
for i in range(0, len(table_list)):
    if re.search(status_pattern, table_list[i].page_content):
        #pages[i].metadata['status'] = 'Yes'
        status_list.append(table_list[i])
    #else:
        #pages[i].metadata['status'] = 'No'
        #other_list.append(pages[i])

status_list

[Document(metadata={'source': 'data.pdf', 'page': 46, 'label': 'Table'}, page_content='21009_Reconstruction of CCKWW_Draft EIA Report 47 \nlisted according to the species’ rarity based on Khew, 2015. Global conservation statuses were \nderived from the IUCN Red List of Threatened Species (IUCN, 2021). \n \nTable 4.1. Conservation status for flora & fauna species and their respective definitions, adapted from \nIUCN Red List (2021) and Singapore Red Data Book (Davison, Ng, & Chew, 2008) \nConservation Status Definition \nGlobal \nExtinct (EX) \nThere is no reasonable doubt that the last individual has died. Exhaustive \nsurveys in known and/or expected habitat, at appropriate times, throughout its \nhistoric range have failed to record an individual. Surveys should be over a \ntime frame appropriate to the taxon’s life cycle and life form. \nExtinct in the Wild (EW) \nKnown only to survive in cultivation, in captivity or as a naturalized population \n(or populations) well outside the pa

# rag

In [4]:
# 4s
# retriever
db1 = FAISS.from_documents(table_list, HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2'))
retriever = db1.as_retriever(search_type='mmr', search_kwargs={"k": 10})

In [75]:
RAG_TEMPLATE = """
You are an assistant for question-answering tasks. Answer the following question based only on the provided context.
Some of the context might be relevant to the question, but some might not be.

<context>
{context}
</context>

Answer the following question:

{question}"""

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def rag(message):
    doc = retriever.invoke(message)
    context = format_docs(doc)
    prompt = RAG_TEMPLATE.format(context=context, question=message)
    answer = llm(prompt)
    
    return answer, context

## species name

In [69]:
def get_unique_lines(answer):
    lines = answer.splitlines()
    unique_lines = []
    seen_lines = set()
    for line in lines:
        if line not in seen_lines:
            unique_lines.append(line)
            seen_lines.add(line)
    return unique_lines

In [55]:
len(species_list)

13

In [103]:
context = format_docs(species_list)
prompt = RAG_TEMPLATE.format(context=context, question='Please list all species name in the context, present each name in a new line.')
answer = llm(prompt)
names = get_unique_lines(answer)
names

['Ampelocissus elegans',
 'Ampelocissus gracilis',
 'Aphanamixis polystachya',
 'Archidendron contortum',
 'Archidendron jiringa',
 'Ardisia elliptica',
 'Artabotrys maingayi',
 'Artabotrys suaveolens',
 'Artocarpus lacucha',
 'Baccaurea motleyana',
 'Bridelia stipularis',
 'Callicarpa longifolia',
 'Calophyllum inophyllum',
 'Causonis trifolia',
 'Chassalia curviflora',
 'Cissus repens',
 'Clerodendrum villosum',
 'Cratoxylum maingayi',
 'Cyathea latebrosa',
 'Cyclosorus opulentus',
 'Cyclosorus polycarpus',
 'Cyrtococcum accrescens',
 'Cyrtococcum patens',
 'Dalbergia junghuhnii',
 'Dendrotrophe varians',
 'Dioscorea cf polyclados',
 'Dissochaeta sp.',
 'Dracaena cantleyi',
 'Embelia canescens',
 'Enkleia malaccensis',
 'Eurycoma longifolia',
 'Ficus apiocarpa',
 'Ficus aurata',
 'Ficus sagittata',
 'Ficus superba',
 'Ficus vasculosa',
 'Glochidion zeylanicum var. zeylanicum',
 'Gnetum latifolium',
 'Goniophlebium percussum',
 'Grenacheria amentacea',
 'Guioa pubescens',
 'Gynochthod

In [109]:
# 100*2mins = 200mins = 3.3hrs
def check_names():
    context = format_docs(species_list)
    prompt = RAG_TEMPLATE.format(context=context, question='Please list all species name in the context, present each name in a new line.')
    count = 0
    for i in range(0, 100):
        answer = llm(prompt)
        names = get_unique_lines(answer)
        if names == names_baseline:
            count += 1
    return count

check_names()

100

## conservation status

In [66]:
len(status_list)

4

In [101]:
context = format_docs(status_list)
prompt = RAG_TEMPLATE.format(context=context, question='Please list all conservation status in the context, present each status in a new line.')
answer = llm(prompt)
status = get_unique_lines(answer)
status

['Extinct (EX)',
 'Extinct in the Wild (EW)',
 'Critically Endangered (CR)',
 'Endangered (EN)',
 'Vulnerable (VU)',
 'Near Threatened (NT)',
 'Least Concern (LC)',
 'Data Deficient (DD)',
 'Not Evaluated (NE)',
 'Presumed Nationally Extinct (NE)',
 'Local Critically Endangered (CR)',
 'Local Endangered (EN)',
 'Local Vulnerable (VU)']

In [108]:
# 100*6.5s = 650s = 11mins
def check_status():
    context = format_docs(status_list)
    prompt = RAG_TEMPLATE.format(context=context, question='Please list all conservation status in the context, present each status in a new line.')
    count = 0
    for i in range(0, 100):
        answer = llm(prompt)
        status = get_unique_lines(answer)
        if status == status_baseline:
            count += 1
    return count

check_status()

100

In [89]:
context = format_docs(status_list)
prompt = RAG_TEMPLATE.format(context=context, question='Please list all GLOBAL & LOCAL conservation status in the context, present each status in a new line.')
answer = llm(prompt)
status = get_unique_lines(answer)
status

['Global Conservation Statuses:',
 '1. Extinct (EX)',
 '2. Extinct in the Wild (EW)',
 '3. Critically Endangered (CR)',
 '4. Endangered (EN)',
 '5. Vulnerable (VU)',
 '6. Near Threatened (NT)',
 '7. Least Concern (LC)',
 '8. Data Deficient (DD)',
 '9. Not Evaluated (NE)',
 '',
 'Local Conservation Statuses:',
 '1. Presumed Nationally Extinct (NE)',
 '2. Critically Endangered (CR)',
 '3. Endangered (EN)',
 '4. Vulnerable (VU)']

In [88]:
context = format_docs(status_list)
prompt = RAG_TEMPLATE.format(context=context, question='Please list all LOCAL only conservation status in the context, present each status in a new line.')
answer = llm(prompt)
status = get_unique_lines(answer)
status

['Presumed Nationally Extinct (NE)',
 'Critically Endangered (CR)',
 'Endangered (EN)',
 'Vulnerable (VU)']

In [100]:
context = format_docs(status_list)
prompt = RAG_TEMPLATE.format(context=context, question='Please list all GLOBAL only conservation status in the context, present each status in a new line.')
answer = llm(prompt)
status = get_unique_lines(answer)
status

['Extinct (EX)',
 'Extinct in the Wild (EW)',
 'Critically Endangered (CR)',
 'Endangered (EN)',
 'Vulnerable (VU)',
 'Data Deficient (DD)',
 'Not Evaluated (NE)']

## Remarks

#save dropdown info locally
import pickle

#Save list and dictionary to file
with open('result.pkl', 'wb') as f:
    pickle.dump(names, f)
    pickle.dump(status, f)

In [98]:
import pickle

# Load list and dictionary from file
with open('result.pkl', 'rb') as f:
    names_baseline = pickle.load(f)
    status_baseline = pickle.load(f)